In [23]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

import xgboost as xgb

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    classification_report)

In [24]:
df_train = pd.read_csv("dataset/train.csv")

In [25]:
df_test = pd.read_csv("dataset/test.csv")

In [26]:
df_train.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [27]:
df_train.isna().sum()

id                             0
Age                            0
Annual_Income_USD              0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Environmental_Concern_Level    0
Gender                         0
City_Type                      0
Current_Car_Type               0
Home_Charging_Possible         0
Subsidy_Available              0
Range_Anxiety_Level            0
Will_Buy_EV                    0
dtype: int64

In [28]:
df_train.dtypes

id                               int64
Age                              int64
Annual_Income_USD              float64
Daily_Commute_km               float64
Number_of_Cars_Owned             int64
Charging_Stations_Near_Home      int64
Charging_Stations_Near_Work      int64
Environmental_Concern_Level    float64
Gender                             str
City_Type                          str
Current_Car_Type                   str
Home_Charging_Possible             str
Subsidy_Available                  str
Range_Anxiety_Level                str
Will_Buy_EV                        str
dtype: object

In [29]:
df_test.dtypes

id                               int64
Age                              int64
Annual_Income_USD              float64
Daily_Commute_km               float64
Number_of_Cars_Owned             int64
Charging_Stations_Near_Home      int64
Charging_Stations_Near_Work      int64
Environmental_Concern_Level    float64
Gender                             str
City_Type                          str
Current_Car_Type                   str
Home_Charging_Possible             str
Subsidy_Available                  str
Range_Anxiety_Level                str
dtype: object

In [30]:
df_train.nunique()

id                             668665
Age                                45
Annual_Income_USD               13214
Daily_Commute_km                  805
Number_of_Cars_Owned                4
Charging_Stations_Near_Home        15
Charging_Stations_Near_Work        20
Environmental_Concern_Level         5
Gender                              3
City_Type                           3
Current_Car_Type                    4
Home_Charging_Possible              2
Subsidy_Available                   2
Range_Anxiety_Level                 3
Will_Buy_EV                         2
dtype: int64

In [31]:
binary_cols = [
    'Home_Charging_Possible',
    'Subsidy_Available',
    'Will_Buy_EV'
]

le = LabelEncoder() 
for col in binary_cols: 
    df_train[col] = le.fit_transform(df_train[col].astype(str)).astype(int)
    if col in df_test.columns: 
        df_test[col] = le.transform(df_test[col].astype(str)).astype(int)

In [32]:
onehot_cols = [
    'Gender',
    'City_Type',
    'Current_Car_Type'
]

In [33]:
encoder = OneHotEncoder(
    handle_unknown='ignore', 
    drop='first',
    sparse_output=False)

df_train_encoded = encoder.fit_transform(df_train[onehot_cols]).astype(int)
df_test_encoded = encoder.transform(df_test[onehot_cols]).astype(int)

In [34]:
train_encoded = pd.DataFrame(
    df_train_encoded,
    columns=encoder.get_feature_names_out(onehot_cols),
    index=df_train.index
)

test_encoded = pd.DataFrame(
    df_test_encoded,
    columns=encoder.get_feature_names_out(onehot_cols),
    index=df_test.index
)

In [35]:
df_train = pd.concat(
    [df_train.drop(columns=onehot_cols), train_encoded],
    axis=1
)

df_test = pd.concat(
    [df_test.drop(columns=onehot_cols), test_encoded],
    axis=1
)

In [36]:
mapping = {
    'Low': 0,
    'Medium': 1,
    'High': 2
}

df_train['Range_Anxiety_Level'] = df_train['Range_Anxiety_Level'].map(mapping).astype(int)
df_test['Range_Anxiety_Level'] = df_test['Range_Anxiety_Level'].map(mapping).astype(int)

In [37]:
df_train.dtypes

id                               int64
Age                              int64
Annual_Income_USD              float64
Daily_Commute_km               float64
Number_of_Cars_Owned             int64
Charging_Stations_Near_Home      int64
Charging_Stations_Near_Work      int64
Environmental_Concern_Level    float64
Home_Charging_Possible           int64
Subsidy_Available                int64
Range_Anxiety_Level              int64
Will_Buy_EV                      int64
Gender_Male                      int64
Gender_Other                     int64
City_Type_Suburban               int64
City_Type_Urban                  int64
Current_Car_Type_SUV             int64
Current_Car_Type_Sedan           int64
Current_Car_Type_Truck           int64
dtype: object

In [38]:
df_test.dtypes

id                               int64
Age                              int64
Annual_Income_USD              float64
Daily_Commute_km               float64
Number_of_Cars_Owned             int64
Charging_Stations_Near_Home      int64
Charging_Stations_Near_Work      int64
Environmental_Concern_Level    float64
Home_Charging_Possible           int64
Subsidy_Available                int64
Range_Anxiety_Level              int64
Gender_Male                      int64
Gender_Other                     int64
City_Type_Suburban               int64
City_Type_Urban                  int64
Current_Car_Type_SUV             int64
Current_Car_Type_Sedan           int64
Current_Car_Type_Truck           int64
dtype: object

In [39]:
df_train.describe()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV,Gender_Male,Gender_Other,City_Type_Suburban,City_Type_Urban,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck
count,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000
mean,334332.000000,47.039171,84769.266989,32.158298,1.712626,4.960408,7.176314,2.935477,0.691941,0.627981,0.100031,0.174645,0.550282,0.007902,0.381921,0.432661,0.368712,0.453828,0.058659
std,193027.103211,12.875448,28648.029042,18.730474,0.729275,3.926843,5.186627,1.429119,0.461691,0.483344,0.310784,0.379663,0.497466,0.088543,0.485858,0.495445,0.482456,0.497864,0.234985
min,0.000000,25.000000,30000.000000,5.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,167166.000000,36.000000,67376.000000,17.200000,1.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,334332.000000,47.000000,84880.000000,33.600000,2.000000,4.000000,6.000000,3.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,501498.000000,58.000000,102753.000000,47.400000,2.000000,7.000000,10.000000,4.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000
max,668664.000000,69.000000,188549.000000,98.700000,4.000000,14.000000,19.000000,5.000000,1.000000,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [42]:
df_train.to_parquet("dataset/df_train_clean.parquet", index=False)

In [43]:
df_test.to_parquet("dataset/df_test_clean.parquet", index=False)